# ACT vs ACM 결과 시각화 + 다운로드

> **train.ipynb 학습 완료 후 실행하세요.**

### 이 노트북이 하는 일
- 학습 로스(loss) 곡선 그래프
- ACT vs ACM 성능 비교 (성공률, 보상 등)
- 시뮬레이션 영상 재생
- 결과 파일을 ZIP으로 묶어 다운로드

### 수정할 것
- 없음 (train.ipynb 결과를 자동으로 찾아서 표시)
- 셀 0 → 셀 1 → ... 순서대로 실행하면 됩니다

---
## 0. 경로 설정

학습 결과 폴더를 찾습니다.
- `ACT: OK` / `ACM: OK` → 정상
- `NOT FOUND` → train.ipynb를 먼저 실행하세요

In [1]:
BASE_DIR = "lerobot_project"
VENV_NAME = "lerobot_env"
PYTHON_VERSION = "3.12"
REPO_NAME = "lerobot-ac-mlp"

import os, sys, json, glob
from pathlib import Path

home_dir = os.path.expanduser('~')
base_path = os.path.join(home_dir, BASE_DIR)
venv_path = os.path.join(base_path, VENV_NAME)
site_packages = os.path.join(venv_path, 'lib', f'python{PYTHON_VERSION}', 'site-packages')
if site_packages not in sys.path: sys.path.insert(0, site_packages)
project_dir = os.path.join(base_path, REPO_NAME)

ACT_OUTPUT = os.path.join(project_dir, "outputs/train/act_transfer_cube_test")
ACM_OUTPUT = os.path.join(project_dir, "outputs/train/acm_transfer_cube_test")

for name, p in [("ACT", ACT_OUTPUT), ("ACM", ACM_OUTPUT)]:
    exists = os.path.exists(p)
    print(f"  {name}: {'OK' if exists else 'NOT FOUND'} -> {p}")

  ACT: OK -> /home1/eunji24/lerobot_project/lerobot-ac-mlp/outputs/train/act_transfer_cube_test
  ACM: OK -> /home1/eunji24/lerobot_project/lerobot-ac-mlp/outputs/train/acm_transfer_cube_test


---
## 1. 학습 로스 곡선

학습이 잘 되고 있는지 확인하는 그래프입니다.
- **Loss가 내려가면** → 학습이 진행 중
- **Loss가 수렴하면** → 학습이 완료에 가까움
- 그래프가 `outputs/loss_curves.png`로 저장됩니다

> "로그 파일 없음"이 뜨면 LeRobot 버전에 따라 로그 형식이 다를 수 있습니다. eval 결과(셀 2)로 확인하세요.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (name, output_dir) in enumerate([("ACT", ACT_OUTPUT), ("ACM", ACM_OUTPUT)]):
    log_file = os.path.join(output_dir, "training_log.csv")
    jsonl_file = os.path.join(output_dir, "training_metrics.jsonl")
    
    steps, losses = [], []
    
    # JSONL 형식 시도
    if os.path.exists(jsonl_file):
        with open(jsonl_file) as f:
            for line in f:
                d = json.loads(line)
                if 'loss' in d:
                    steps.append(d.get('step', len(steps)))
                    losses.append(d['loss'])
    
    # CSV 형식 시도
    elif os.path.exists(log_file):
        import csv
        with open(log_file) as f:
            reader = csv.DictReader(f)
            for row in reader:
                if 'loss' in row:
                    steps.append(int(row.get('step', len(steps))))
                    losses.append(float(row['loss']))
    
    # train_info에서 시도
    if not losses:
        info_files = sorted(Path(output_dir).rglob("train_info.json")) + \
                     sorted(Path(output_dir).rglob("training_state.json"))
        if info_files:
            with open(info_files[0]) as f:
                data = json.load(f)
            if isinstance(data, dict) and 'log_history' in data:
                for entry in data['log_history']:
                    if 'loss' in entry:
                        steps.append(entry.get('step', len(steps)))
                        losses.append(entry['loss'])
    
    ax = axes[idx]
    if losses:
        ax.plot(steps, losses, linewidth=0.8)
        ax.set_xlabel('Step')
        ax.set_ylabel('Loss')
        ax.set_title(f'{name} Training Loss')
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'{name}\n로그 파일 없음', ha='center', va='center', fontsize=14)
        ax.set_title(f'{name}')

plt.tight_layout()
fig_path = os.path.join(project_dir, 'outputs/loss_curves.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"저장: {fig_path}")

---
## 2. Eval 성능 비교

ACT와 ACM의 시뮬레이션 평가 결과를 비교합니다.

### 주요 지표
| 지표 | 의미 | 좋은 값 |
|------|------|---------|
| `pc_success` | 성공률 (0~1) | 높을수록 좋음 |
| `avg_sum_reward` | 평균 누적 보상 | 높을수록 좋음 |
| `avg_max_reward` | 평균 최대 보상 | 높을수록 좋음 |

- 첫 번째 셀: 숫자 테이블
- 두 번째 셀: 막대 그래프 (`outputs/eval_comparison.png` 저장)

In [3]:
import json
from pathlib import Path

results = {}

for name, output_dir in [("ACT", ACT_OUTPUT), ("ACM", ACM_OUTPUT)]:
    # eval_info.json 찾기
    eval_jsons = sorted(Path(output_dir).rglob("eval_info.json"))
    if eval_jsons:
        with open(eval_jsons[-1]) as f:
            data = json.load(f)
        results[name] = data.get("overall", data)
    else:
        print(f"[{name}] eval_info.json 없음")

if results:
    print(f"{'Metric':<25} {'ACT':>10} {'ACM':>10}")
    print("-" * 47)
    
    all_keys = set()
    for v in results.values():
        all_keys.update(v.keys())
    all_keys.discard('video_paths')
    
    for k in sorted(all_keys):
        act_val = results.get('ACT', {}).get(k, 'N/A')
        acm_val = results.get('ACM', {}).get(k, 'N/A')
        
        if isinstance(act_val, float):
            act_str = f"{act_val:.4f}"
        else:
            act_str = str(act_val)
        
        if isinstance(acm_val, float):
            acm_str = f"{acm_val:.4f}"
        else:
            acm_str = str(acm_val)
        
        print(f"{k:<25} {act_str:>10} {acm_str:>10}")
else:
    print("eval 결과가 없습니다. train.ipynb를 먼저 실행하세요.")

[ACT] eval_info.json 없음
[ACM] eval_info.json 없음
eval 결과가 없습니다. train.ipynb를 먼저 실행하세요.


In [4]:
# 막대 그래프 비교
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

if results and len(results) == 2:
    metrics = [k for k in sorted(results['ACT'].keys()) 
               if k != 'video_paths' and isinstance(results['ACT'].get(k), (int, float))]
    
    if metrics:
        act_vals = [float(results['ACT'].get(m, 0)) for m in metrics]
        acm_vals = [float(results['ACM'].get(m, 0)) for m in metrics]
        
        x = np.arange(len(metrics))
        width = 0.35
        
        fig, ax = plt.subplots(figsize=(10, 5))
        bars1 = ax.bar(x - width/2, act_vals, width, label='ACT', color='#4C72B0')
        bars2 = ax.bar(x + width/2, acm_vals, width, label='ACM', color='#DD8452')
        
        ax.set_xlabel('Metric')
        ax.set_ylabel('Value')
        ax.set_title('ACT vs ACM Evaluation')
        ax.set_xticks(x)
        ax.set_xticklabels(metrics, rotation=30, ha='right')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        # 값 표시
        for bar in bars1:
            ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                        ha='center', va='bottom', fontsize=8)
        for bar in bars2:
            ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                        ha='center', va='bottom', fontsize=8)
        
        plt.tight_layout()
        fig_path = os.path.join(project_dir, 'outputs/eval_comparison.png')
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"저장: {fig_path}")
else:
    print("비교할 결과가 부족합니다.")

비교할 결과가 부족합니다.


---
## 3. 영상 재생

시뮬레이션 에피소드 영상을 확인합니다.
- 첫 번째 셀: 영상 목록 + 용량
- 두 번째 셀: ACT/ACM 각각 첫 번째 영상 재생
- Jupyter에서 바로 재생되며, 안 되면 파일 경로가 출력됩니다

In [ ]:
from pathlib import Path

all_videos = []
for name, output_dir in [("ACT", ACT_OUTPUT), ("ACM", ACM_OUTPUT)]:
    videos = sorted(Path(output_dir).rglob("*.mp4"))
    print(f"\n[{name}] 영상 {len(videos)}개")
    for v in videos:
        size_mb = v.stat().st_size / 1024 / 1024
        print(f"  {v.relative_to(output_dir)} ({size_mb:.1f} MB)")
        all_videos.append((name, v))

if not all_videos:
    print("\n영상이 없습니다.")

In [ ]:
# 첫 번째 영상씩 재생
from IPython.display import Video, display, HTML
from pathlib import Path

shown = set()
for name, output_dir in [("ACT", ACT_OUTPUT), ("ACM", ACM_OUTPUT)]:
    videos = sorted(Path(output_dir).rglob("*.mp4"))
    if videos:
        v = videos[0]
        print(f"\n--- {name}: {v.name} ---")
        try:
            display(Video(str(v), embed=True, width=480))
        except Exception as e:
            print(f"  재생 실패: {e}")
            print(f"  파일 경로: {v}")

---
## 4. 결과 다운로드 (ZIP)

모든 결과를 ZIP 파일로 묶어 로컬로 다운로드합니다.

### 설정
| 변수 | 기본값 | 설명 |
|------|--------|------|
| `INCLUDE_CHECKPOINTS` | `True` | 모델 가중치 (용량 큼, 불필요시 `False`) |
| `INCLUDE_VIDEOS` | `True` | 시뮬레이션 영상 |
| `INCLUDE_EVAL` | `True` | 평가 결과 JSON |
| `INCLUDE_PLOTS` | `True` | 그래프 이미지 |

- 첫 번째 셀: ZIP 파일 생성
- 두 번째 셀: 다운로드 링크 (또는 `scp` 명령어 안내)

In [ ]:
# ---------- 설정 ----------
INCLUDE_CHECKPOINTS = True    # 체크포인트 포함 (용량 큼)
INCLUDE_VIDEOS = True         # 영상 포함
INCLUDE_EVAL = True           # eval 결과 JSON 포함
INCLUDE_PLOTS = True          # 그래프 이미지 포함
# --------------------------

import zipfile, shutil
from pathlib import Path

zip_path = os.path.join(project_dir, 'outputs/results.zip')
outputs_dir = os.path.join(project_dir, 'outputs')

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    count = 0
    
    for name, output_dir in [("ACT", ACT_OUTPUT), ("ACM", ACM_OUTPUT)]:
        p = Path(output_dir)
        if not p.exists():
            continue
        
        # Eval JSON
        if INCLUDE_EVAL:
            for f in p.rglob("*.json"):
                arcname = f"{name}/{f.relative_to(p)}"
                zf.write(f, arcname)
                count += 1
        
        # 영상
        if INCLUDE_VIDEOS:
            for f in p.rglob("*.mp4"):
                arcname = f"{name}/{f.relative_to(p)}"
                zf.write(f, arcname)
                count += 1
        
        # 체크포인트
        if INCLUDE_CHECKPOINTS:
            for f in p.rglob("checkpoints/**/*"):
                if f.is_file():
                    arcname = f"{name}/{f.relative_to(p)}"
                    zf.write(f, arcname)
                    count += 1
    
    # 그래프
    if INCLUDE_PLOTS:
        for png in Path(outputs_dir).glob("*.png"):
            zf.write(png, f"plots/{png.name}")
            count += 1

size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"ZIP 생성 완료: {zip_path}")
print(f"  파일 수: {count}개")
print(f"  크기: {size_mb:.1f} MB")

In [ ]:
# Jupyter에서 다운로드 링크 생성
from IPython.display import display, HTML, FileLink

try:
    # FileLink로 다운로드
    rel_path = os.path.relpath(zip_path)
    display(FileLink(rel_path, result_html_prefix='다운로드: '))
except Exception:
    print(f"다운로드 경로: {zip_path}")
    print("터미널에서 scp로 다운로드:")
    print(f"  scp <서버>:{zip_path} .")

In [ ]:
from pathlib import Path
import os

project_dir = os.path.expanduser("~/lerobot_project/lerobot-ac-mlp")

targets = [
    os.path.join(project_dir, "outputs/train/act_transfer_cube_test"),
    os.path.join(project_dir, "outputs/train/acm_transfer_cube_test"),
]

for t in targets:
    print("\n====", t, "====")
    p = Path(t)
    if not p.exists():
        print("폴더 자체가 없음")
        continue

    eval_jsons = sorted(p.rglob("eval_info.json"))
    if eval_jsons:
        for f in eval_jsons:
            print(f)
    else:
        print("eval_info.json 없음")
        